In [11]:
# Set GPU
import os
os.environ["CUDA_VISIBLE_DEVICES"]="2"

In [12]:
from torch.utils.data import Dataset
import os, random, time
import cv2 as cv2
from matplotlib import pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np

import torch
import torchvision
from torchvision import transforms
from torchvision import io
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection import FasterRCNN, fasterrcnn_resnet50_fpn_v2, fasterrcnn_resnet50_fpn, fasterrcnn_mobilenet_v3_large_320_fpn
from torchvision.models.detection.rpn import AnchorGenerator
import torchvision.transforms.functional as F_vision
from torchvision.utils import make_grid
from torch.utils.data import DataLoader, Dataset
from torch.utils.data.sampler import SequentialSampler
from torchvision.models.detection.backbone_utils import resnet_fpn_backbone

from tqdm.notebook import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import albumentations.pytorch
import pandas as pd
from PIL import Image
import imageio
import math
from pathlib import Path
from os.path import isfile, join
import xml.etree.ElementTree as ET
import pandas as pd
import matplotlib.pyplot as plt
from torchmetrics.detection import MeanAveragePrecision
import json
import glob
import re

from config import *
from helpers import *

In [13]:
# Input
folder_models = 'models'
folder_data = 'data'

In [14]:
# Loop over model files
all_results = {}
for file_model in glob.glob(f'{folder_models}/*/modelo*.bin'):
    print(file_model)
    
    # Prepare model
    model, device, bbox_params = prepare_model(file_model, verbose=False)
    
    # Predict val set
    all_bboxes_true, all_bboxes_pred, all_scores = predict_val(folder_data, bbox_params, model, device)
    
    # Evaluate
    results = evaluate(all_bboxes_true, all_bboxes_pred, all_scores)
        
    # Append
    key = file_model.split('/')[-2]
    all_results[key] = results


models/0070_fasterrcnn_resnet50_fpn_th50/modelo.bin


/home/jorism/.miniconda3/envs/pytorch_py312/lib/python3.12/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Encountered more than 100 detections in a single image. This means that certain detections with the lowest scores will be ignored, that may have an undesirable impact on performance. Please consider adjusting the `max_detection_threshold` to suit your use case. To disable this warning, set attribute class `warn_on_many_detections=False`, after initializing the metric.
  warnings.warn(*args, **kwargs)



models/0094_fasterrcnn_resnet50_fpn_deblurGAN/modelo.bin

models/0096_resnet18_fpn_th50/modelo.bin

models/0103_resnet152_fpn_th50/modelo.bin

models/0104_resnet152_fpn_th50_deblurGAN/modelo.bin

models/0105_resnet18_fpn_th50_deblurGAN/modelo_7752.bin


In [17]:
# all_results

In [26]:
# Backbone Confidence Threshold Refinement Step deblurGAN mAP mAR
rows = []
for key in all_results:
    row = {}
    match = re.search(r"resnet[0-9]+", key)
    if match:
        backbone_name = match.group()
        row['backbone'] = (backbone_name)

    row['n_layers'] = int(backbone_name[6:])
    row['Confidence Threshold'] = 0.5
    row['Refinement Step'] = 'Yes'
    if 'deblurGAN' in key:
        row['deblurGAN'] = 'Yes'
    else:
        row['deblurGAN'] = 'No'

    row['mAP'] = all_results[key]['map']

    rows.append(row)

pd.DataFrame(rows).sort_values(['n_layers', 'deblurGAN'])

,backbone,n_layers,Confidence Threshold,Refinement Step,deblurGAN,mAP
2,resnet18,18,0.5,Yes,No,0.902809
5,resnet18,18,0.5,Yes,Yes,0.826668
0,resnet50,50,0.5,Yes,No,0.927313
1,resnet50,50,0.5,Yes,Yes,0.647665
3,resnet152,152,0.5,Yes,No,0.872988
4,resnet152,152,0.5,Yes,Yes,0.815298


In [27]:
# Write results to file
with open("results.json", "w") as f:
    json.dump(all_results, f, indent=2)